# Factor-analysis dimensionality of evoked variability vs LDA 1
Evoked window (0-0.2 s), single-trial activity **GLM-residualized** (stimulus side, contrast, choice, feedback removed). Per session x region, **Factor Analysis** (n-factors chosen by cross-validation) separates shared from private variability, yielding **% shared variance** and **shared dimensionality**. Related to LDA 1 per region and with a pooled mixed model.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle, os
from collections import defaultdict
from iblatlas.regions import BrainRegions
from sklearn.decomposition import FactorAnalysis
from sklearn.model_selection import cross_val_score
from scipy.stats import pearsonr, spearmanr
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid'); plt.rcParams['figure.facecolor'] = 'white' 

In [ ]:
prefix = '/home/ines/repositories/representation_learning_variability/paper-individuality/'
firing_rates_dir = prefix + 'data/firing_rates/'
clustering_dir = prefix + 'clustering/'
trials_path = prefix + '4_mice/all_trials_04-05-2026'

WINDOW = (0.0, 0.2)          # evoked window (s)
REGION_LEVEL = 'cosmos'      # 'cosmos' (broad) or 'beryl' (fine)
DROP = ['root', 'void']
MIN_NEURONS = 20             # per-region neurons (variance>0) needed in a session
MIN_TRIALS = 50
MIN_SESSIONS_POOLED = 15     # regions in >= this many sessions enter the pooled model
FACTOR_GRID = [1, 2, 3, 5, 8]   # candidate #factors (CV-selected)
CV = 3
N_REPEATS = 3                # neuron-subsampling repeats (averaged)
SEED = 0

lda = pd.read_pickle(clustering_dir + 'mouse_LDA_5_bins_cut19-06-2026').rename(columns={0: 'lda_1'})
trials_df = pd.read_parquet(trials_path)
trial_beh = {s: g.set_index('trial_id')[['choice', 'correct']] for s, g in trials_df.groupby('session')}
print("params set")

## Load evoked single-trial rates + region labels

In [ ]:
pkl_files = sorted([f for f in os.listdir(firing_rates_dir) if f.startswith('firing_rate_')])
with open(os.path.join(firing_rates_dir, pkl_files[0]), 'rb') as f:
    s0 = pickle.load(f)
tcols = sorted([c for c in s0.columns if c.startswith('t_')], key=lambda x: float(x.split('_')[1]))
tsec = np.array([float(c.split('_')[1]) for c in tcols])
wcols = [c for c, m in zip(tcols, (tsec >= WINDOW[0]) & (tsec <= WINDOW[1])) if m]
print(f"Window {WINDOW}: {len(wcols)} of {len(tcols)} bins")

sess_pivots = defaultdict(list); sess_cond = {}
for i, fn in enumerate(pkl_files):
    try:
        with open(os.path.join(firing_rates_dir, fn), 'rb') as f:
            d = pickle.load(f)
        d = d[~d['area'].isin(DROP)]
        if len(d) == 0: continue
        d = d.copy()
        d['rate'] = np.nanmean(d[wcols].values, axis=1)
        d['nuid'] = d['pid'].astype(str) + '__' + d['neuron_id'].astype(str)
        sess_pivots[d['session'].iloc[0]].append(
            (d.pivot_table(index='trial_id', columns='nuid', values='rate'),
             d.groupby('nuid')['area'].first()))
        if d['session'].iloc[0] not in sess_cond:
            sess_cond[d['session'].iloc[0]] = d.drop_duplicates('trial_id').set_index('trial_id')['condition']
        if (i + 1) % 100 == 0: print(f"  {i+1}/{len(pkl_files)} files...")
    except Exception as e:
        print(f"Error {fn}: {e}")

session_X, session_area = {}, {}
for s, parts in sess_pivots.items():
    X = pd.concat([p for p, _ in parts], axis=1)
    a = pd.concat([am for _, am in parts]); a = a[~a.index.duplicated()]
    session_X[s] = X; session_area[s] = a.reindex(X.columns)

if REGION_LEVEL == 'cosmos':
    br = BrainRegions()
    all_areas = pd.unique(np.concatenate([a.dropna().values for a in session_area.values()]))
    cmap = dict(zip(all_areas, br.acronym2acronym(all_areas, mapping='Cosmos')))
    session_area = {s_: a.map(cmap).where(lambda x: ~x.isin(DROP)) for s_, a in session_area.items()}
print(f"Sessions loaded: {len(session_X)}")

## GLM-residualize each session (remove stimulus + choice + feedback)

In [ ]:
session_R = {}
for s, X in session_X.items():
    cond = sess_cond[s].reindex(X.index).astype(str)
    side, contrast = cond.str.split('_').str[0], cond.str.split('_').str[1]
    beh = trial_beh.get(s)
    if beh is None: continue
    choice = beh['choice'].reindex(X.index).astype(str)
    feedback = beh['correct'].reindex(X.index).astype(str)
    meta = pd.DataFrame({'side': side, 'contrast': contrast, 'choice': choice, 'feedback': feedback}, index=X.index)
    valid = ~meta.isin(['nan', 'None']).any(axis=1)
    Xv, meta = X.loc[valid], meta.loc[valid]
    if Xv.shape[0] < MIN_TRIALS: continue
    D = pd.get_dummies(meta.astype(str), drop_first=False).astype(float); D['intercept'] = 1.0
    R = Xv.values.astype(float) - D.values @ np.linalg.lstsq(D.values, Xv.values.astype(float), rcond=None)[0]
    session_R[s] = pd.DataFrame(R, index=Xv.index, columns=Xv.columns)
print(f"Residualized sessions: {len(session_R)}")

# region -> eligible sessions (>= MIN_NEURONS variance>0 residual neurons)
region_sessions = defaultdict(list)
for s, R in session_R.items():
    areas = session_area[s]
    valid = R.columns[(R.std(axis=0) > 0).values]
    for region in areas.loc[valid].dropna().unique():
        n = int((areas.loc[valid] == region).sum())
        if n >= MIN_NEURONS:
            region_sessions[region].append((s, n))
print("Regions (>= %d sessions):" % MIN_SESSIONS_POOLED,
      [r for r, v in region_sessions.items() if len(v) >= MIN_SESSIONS_POOLED])

## Factor analysis: % shared variance & shared dimensionality

In [ ]:
def fa_metrics(mat, grid, cv):
    """mat: trials x neurons (residuals). Returns (%shared var, shared dimensionality, n_factors)."""
    sd = mat.std(0); mat = mat[:, sd > 0]
    n_tr, n_ne = mat.shape
    if n_ne < 3: return np.nan, np.nan, np.nan
    Xz = (mat - mat.mean(0)) / mat.std(0)
    ks = [k for k in grid if k < min(n_ne, n_tr)]
    if not ks: return np.nan, np.nan, np.nan
    best_k, best_ll = ks[0], -np.inf
    for k in ks:
        try:
            ll = cross_val_score(FactorAnalysis(n_components=k, max_iter=1000), Xz, cv=cv).mean()
        except Exception:
            ll = -np.inf
        if ll > best_ll: best_ll, best_k = ll, k
    fa = FactorAnalysis(n_components=best_k, max_iter=1000).fit(Xz)
    L = fa.components_.T                      # neurons x k
    shared = L @ L.T
    psi = fa.noise_variance_
    sv = np.clip(np.diag(shared), 0, None)
    pct_sv = float(np.mean(sv / (sv + psi)))
    ev = np.sort(np.clip(np.linalg.eigvalsh(shared), 0, None))[::-1]
    d_shared = float((ev.sum() ** 2) / (ev ** 2).sum()) if ev.sum() > 0 else np.nan
    return pct_sv, d_shared, best_k

rng = np.random.default_rng(SEED)
regions = [r for r, v in region_sessions.items() if len(v) >= MIN_SESSIONS_POOLED]
records = []
for region in regions:
    sess = region_sessions[region]
    N_common = min(n for _, n in sess)
    for s, navail in sess:
        R = session_R[s]; areas = session_area[s]
        cols = R.columns[((areas == region) & (R.std(axis=0) > 0)).values]
        Rr = R[cols].dropna(axis=0, how='any')
        if Rr.shape[0] < MIN_TRIALS or Rr.shape[1] < N_common: continue
        psv, dsh = [], []
        for _ in range(N_REPEATS):
            sub = rng.choice(Rr.columns.values, N_common, replace=False)
            a, b, k = fa_metrics(Rr[sub].values, FACTOR_GRID, CV)
            psv.append(a); dsh.append(b)
        records.append(dict(region=region, session=s, N_common=N_common,
                            pct_sv=np.nanmean(psv), d_shared=np.nanmean(dsh)))

fa_df = (pd.DataFrame(records)
         .merge(lda[['session', 'lda_1', 'mouse_name']], on='session', how='left')
         .dropna(subset=['pct_sv', 'd_shared', 'lda_1', 'mouse_name']))
print(f"FA computed: {len(fa_df)} region-session entries | {fa_df['region'].nunique()} regions | "
      f"{fa_df['session'].nunique()} sessions | {fa_df['mouse_name'].nunique()} mice")
print(fa_df.groupby('region').agg(n=('session','size'), N=('N_common','first'),
      mean_pct_sv=('pct_sv','mean'), mean_d_shared=('d_shared','mean')).to_string())

## Per-region correlations + pooled mixed model

In [ ]:
# per-region correlations
for metric, label in [('pct_sv', '% shared variance'), ('d_shared', 'shared dimensionality')]:
    print(f"\n--- {label} vs LDA 1 (per region) ---")
    for region, g in fa_df.groupby('region'):
        r, p = pearsonr(g['lda_1'], g[metric]); rho, pp = spearmanr(g['lda_1'], g[metric])
        print(f"  {region:10s} n={len(g):3d}: pearson r={r:+.2f} p={p:.3f} | spearman rho={rho:+.2f} p={pp:.3f}")

# pooled mixed model: metric ~ lda_1 + C(region) + (1|mouse) [session nested]
def fit_pooled(metric):
    try:
        return smf.mixedlm(f"{metric} ~ lda_1 + C(region)", fa_df, groups=fa_df['mouse_name'],
                           vc_formula={'session': '0 + C(session)'}).fit(reml=True), 'mouse+session'
    except Exception:
        return smf.mixedlm(f"{metric} ~ lda_1 + C(region)", fa_df, groups=fa_df['session']).fit(reml=True), 'session'

for metric, label in [('pct_sv', '% shared variance'), ('d_shared', 'shared dimensionality')]:
    r, model = fit_pooled(metric)
    b, se, p = r.params['lda_1'], r.bse['lda_1'], r.pvalues['lda_1']
    print("\n" + "="*66)
    print(f"POOLED: {metric} ~ lda_1 + C(region) + (1|mouse)   [RE: {model}]")
    print(f"  lda_1 coef = {b:+.4f}  SE = {se:.4f}  z = {b/se:+.2f}  p = {p:.4f}")

In [ ]:
fig, axes = plt.subplots(2, len(fa_df['region'].unique()),
                         figsize=(3.4 * fa_df['region'].nunique(), 7.5), squeeze=False)
regions_sorted = fa_df.groupby('region')['session'].size().sort_values(ascending=False).index.tolist()
for row, (metric, ylab) in enumerate([('pct_sv', '% shared var'), ('d_shared', 'shared dim.')]):
    for k, region in enumerate(regions_sorted):
        ax = axes[row][k]
        g = fa_df[fa_df['region'] == region]
        x, y = g['lda_1'].values, g[metric].values
        r, p = pearsonr(x, y)
        ax.scatter(x, y, c=x, cmap='coolwarm', s=35, alpha=0.8, edgecolors='black', linewidth=0.3)
        if len(g) >= 3:
            z = np.polyfit(x, y, 1); xl = np.linspace(x.min(), x.max(), 50)
            ax.plot(xl, np.polyval(z, xl), 'k-', lw=1.8)
        ax.set_title(f'{region} (n={len(g)})\nr={r:.2f} p={p:.3f}', fontsize=8)
        ax.set_xlabel('LDA 1'); ax.set_ylabel(ylab, fontsize=8)
        sns.despine(ax=ax, offset=3)
fig.suptitle('Factor-analysis evoked variability metrics vs LDA 1', y=1.01, fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()